# rStar-Math: Linear Algebra Examples

This notebook demonstrates how rStar-Math handles linear algebra problems with visualizations.

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from src.core.mcts import MCTS
from src.core.ppm import ProcessPreferenceModel
from src.models.model_interface import ModelFactory

In [ ]:
# Initialize components
mcts = MCTS.from_config_file('config/default.json')
ppm = ProcessPreferenceModel.from_config_file('config/default.json')
model = ModelFactory.create_model('openai', os.getenv('OPENAI_API_KEY'), 'config/default.json')

## 1. Matrix Operations

In [ ]:
matrix_problems = [
    "Find the determinant of [[1, 2], [3, 4]]",
    "Solve the system of equations: 2x + y = 5, x - y = 1",
    "Find the eigenvalues of [[2, 1], [1, 2]]",
    "Calculate the inverse of [[1, 2], [3, 4]]"
]

def visualize_matrix(matrix_str: str):
    """Visualize matrix as a heatmap."""
    matrix = np.array(eval(matrix_str))
    plt.figure(figsize=(8, 6))
    plt.imshow(matrix, cmap='viridis')
    plt.colorbar()
    for i in range(matrix.shape[0]):
        for j in range(matrix.shape[1]):
            plt.text(j, i, f'{matrix[i,j]:.2f}', ha='center', va='center')
    plt.title('Matrix Visualization')
    plt.show()

for problem in matrix_problems:
    print(f"Problem: {problem}\n")
    action, trajectory = mcts.search(problem)
    
    print("Solution Steps:")
    for step in trajectory:
        confidence = ppm.evaluate_step(step['state'], model)
        print(f"- {step['state']}")
        print(f"  Confidence: {confidence:.2f}\n")
        
    # Visualize matrix if present in problem
    if '[[' in problem:
        matrix_str = problem[problem.find('[['):problem.find(']]')+2]
        visualize_matrix(matrix_str)
    print("-" * 50 + "\n")

## 2. Vector Spaces and Transformations

In [ ]:
def plot_vector_transformation(matrix: np.ndarray):
    """Visualize linear transformation."""
    fig = plt.figure(figsize=(12, 5))
    
    # Original vectors
    ax1 = fig.add_subplot(121)
    vectors = np.array([[1, 0], [0, 1]])
    ax1.quiver([0, 0], [0, 0], vectors[:, 0], vectors[:, 1],
               angles='xy', scale_units='xy', scale=1)
    ax1.set_xlim(-2, 2)
    ax1.set_ylim(-2, 2)
    ax1.grid(True)
    ax1.set_title('Original Vectors')
    
    # Transformed vectors
    ax2 = fig.add_subplot(122)
    transformed = np.dot(vectors, matrix)
    ax2.quiver([0, 0], [0, 0], transformed[:, 0], transformed[:, 1],
               angles='xy', scale_units='xy', scale=1)
    ax2.set_xlim(-2, 2)
    ax2.set_ylim(-2, 2)
    ax2.grid(True)
    ax2.set_title('Transformed Vectors')
    
    plt.show()

# Example transformations
transformations = [
    np.array([[2, 0], [0, 2]]),  # Scaling
    np.array([[0, -1], [1, 0]]),  # Rotation
    np.array([[1, 1], [0, 1]])   # Shear
]

for matrix in transformations:
    print(f"Transformation Matrix:\n{matrix}\n")
    plot_vector_transformation(matrix)

## 3. Eigenvalues and Eigenvectors

In [ ]:
def plot_eigenvectors(matrix: np.ndarray):
    """Visualize eigenvectors and their transformations."""
    eigenvals, eigenvecs = np.linalg.eig(matrix)
    
    plt.figure(figsize=(8, 8))
    
    # Plot original vectors
    for i, vec in enumerate(eigenvecs.T):
        plt.quiver(0, 0, vec[0], vec[1], angles='xy', scale_units='xy',
                  scale=1, color='blue', label=f'Eigenvector {i+1}')
        
    # Plot transformed vectors
    transformed = np.dot(matrix, eigenvecs)
    for i, vec in enumerate(transformed.T):
        plt.quiver(0, 0, vec[0], vec[1], angles='xy', scale_units='xy',
                  scale=1, color='red', label=f'Transformed {i+1}')
    
    plt.xlim(-2, 2)
    plt.ylim(-2, 2)
    plt.grid(True)
    plt.legend()
    plt.title('Eigenvectors and Their Transformations')
    plt.show()
    
    print("Eigenvalues:")
    for i, val in enumerate(eigenvals):
        print(f"λ{i+1} = {val:.2f}")

# Example matrices
matrices = [
    np.array([[2, 1], [1, 2]]),  # Symmetric matrix
    np.array([[0, -1], [1, 0]]),  # Rotation matrix
    np.array([[3, 1], [0, 2]])   # Upper triangular matrix
]

for matrix in matrices:
    print(f"\nMatrix:\n{matrix}")
    plot_eigenvectors(matrix)